# 06 — Distributed Computing Concepts

Demonstrates and explains lazy evaluation, the DAG, stage metrics, and the caching/
persistence strategy used in this project  "demonstrate
understanding of distributed computing concepts" Technical Requirement, and directly
useful material for report's "Algorithmic Efficiency" metric.


In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['HADOOP_HOME'] = r'C:\Hadoop'
os.makedirs(r'C:\spark-tmp', exist_ok=True)

In [5]:

try:
    spark.stop()
    print("Stopped existing SparkSession.")
except NameError:
    print("No existing SparkSession found — starting fresh.")
except Exception as e:
    print("Note while stopping old session:", e)


import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['HADOOP_HOME'] = r'C:\Hadoop'

SPARK_TMP_DIR = r'C:\spark-tmp'
os.makedirs(SPARK_TMP_DIR, exist_ok=True)

# Confirm the temp dir exists and is writable before going further
assert os.path.exists(SPARK_TMP_DIR), f"Failed to create {SPARK_TMP_DIR}"
assert os.access(SPARK_TMP_DIR, os.W_OK), f"{SPARK_TMP_DIR} is not writable"
print(f"Using local temp dir: {SPARK_TMP_DIR} (exists & writable)")
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time
import warnings
warnings.filterwarnings("ignore")

spark = SparkSession.builder \
    .appName("BusBunching_06_DistributedConcepts") \
    .master("local[4]") \
    .config("spark.ui.showConsoleProgress", "false") \
    .config("spark.local.dir", SPARK_TMP_DIR) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
actual_temp_dir = spark.sparkContext._temp_dir
print("Spark local dir in use:", actual_temp_dir)
if "spark-tmp" not in actual_temp_dir:
    print("WARNING: Spark is NOT using the configured temp dir. "
          "Restart the Jupyter kernel completely (Kernel > Restart Kernel), "
          "then rerun this cell as the FIRST cell in the notebook.")
location_clean = spark.read.option("header", True).option("inferSchema", True).csv("data/cleaned_locations.csv")
print("Loaded location_clean:", location_clean.count(), "rows")
location_clean.printSchema()

Stopped existing SparkSession.
Using local temp dir: C:\spark-tmp (exists & writable)
Spark local dir in use: C:\spark-tmp\spark-0783a11e-47bf-415a-ae88-5bc26ede62b0\pyspark-61efd4eb-4a31-40d6-836b-1214c971d963
Loaded location_clean: 762 rows
root
 |-- poll_timestamp: timestamp (nullable = true)
 |-- recorded_at_time: timestamp (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- vehicle_ref: string (nullable = true)
 |-- operator_ref: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- bearing: integer (nullable = true)



## 6.1 Lazy evaluation demonstration
Transformations (`.filter()`, `.groupBy()`, `.withColumn()`) build up a logical plan but do
**not** execute until an action (`.show()`, `.count()`, `.collect()`) is called. We show this
by timing the "transformation" step (near-instant) versus the "action" step (does the real work).


In [6]:
start = time.time()
transformed = location_clean.filter(F.col("line_ref").isNotNull()).groupBy("line_ref").count()
print(f"Transformation defined in {time.time()-start:.4f}s (nothing executed yet — lazy)")

start = time.time()
transformed.show()   # this triggers the action, and the actual Spark job runs now
print(f"Action executed in {time.time()-start:.4f}s")


Transformation defined in 0.0500s (nothing executed yet — lazy)
+--------+-----+
|line_ref|count|
+--------+-----+
|     125|   21|
|     17A|    5|
|       7|    3|
|      51|    1|
|      1A|   16|
|      15|    2|
|      11|    1|
|     685|    4|
|      29|    1|
|      42|    6|
|     534|    1|
|     43A|    1|
|     10A|   67|
|       3|    8|
|      30|   18|
|     16A|    1|
|      59|    8|
|      22|    2|
|    217A|    2|
|      52|    2|
+--------+-----+
only showing top 20 rows

Action executed in 2.3133s


+--------+-----+
|line_ref|count|
+--------+-----+
|     125|   21|
|     17A|    5|
|       7|    3|
|      51|    1|
|      1A|   16|
|      15|    2|
|      11|    1|
|     685|    4|
|      29|    1|
|      42|    6|
|     534|    1|
|     43A|    1|
|     10A|   67|
|       3|    8|
|      30|   18|
|     16A|    1|
|      59|    8|
|      22|    2|
|    217A|    2|
|      52|    2|
+--------+-----+
only showing top 20 rows
Action executed in 2.0531s


## 6.2 Inspecting the DAG (logical & physical plan)

In [7]:
transformed.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Filter (2)
            +- Scan csv  (1)


(1) Scan csv 
Output [1]: [line_ref#65]
Batched: false
Location: InMemoryFileIndex [file:/C:/Users/ACER/Desktop/data/cleaned_locations.csv]
PushedFilters: [IsNotNull(line_ref)]
ReadSchema: struct<line_ref:string>

(2) Filter
Input [1]: [line_ref#65]
Condition : isnotnull(line_ref#65)

(3) HashAggregate
Input [1]: [line_ref#65]
Keys [1]: [line_ref#65]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#111L]
Results [2]: [line_ref#65, count#112L]

(4) Exchange
Input [2]: [line_ref#65, count#112L]
Arguments: hashpartitioning(line_ref#65, 200), ENSURE_REQUIREMENTS, [plan_id=172]

(5) HashAggregate
Input [2]: [line_ref#65, count#112L]
Keys [1]: [line_ref#65]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#101L]
Results [2]: [line_ref#65, count(1)#101L AS count#102L]

(6) AdaptiveSparkPlan
Output [2]: 

## 6.3 Stage metrics

In [9]:
# Run a slightly heavier operation and inspect stage info via the SparkContext status tracker
job_result = location_clean.groupBy("line_ref").agg(F.count("*")).collect()

status = spark.sparkContext.statusTracker()
job_ids = status.getJobIdsForGroup()
print("Job IDs executed so far:", job_ids)
for jid in job_ids[-3:]:
    info = status.getJobInfo(jid)
    if info is not None:
        print(f"Job {jid}: status={info.status}, stageIds={info.stageIds}")



Job IDs executed so far: [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
Job 2: status=SUCCEEDED, stageIds=[I@522f232c
Job 1: status=SUCCEEDED, stageIds=[I@2bab4a25
Job 0: status=SUCCEEDED, stageIds=[I@1e922996
